In [2]:
%pip install pandas

Note: you may need to restart the kernel to use updated packages.


In [3]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("data.sqlite")

pd.read_sql("""
SELECT name
FROM sqlite_master
WHERE type = 'table';
""", conn)

,name
0,orderdetails
1,payments
2,offices
3,customers
4,orders
5,productlines
6,products
7,employees


In [4]:
pd.read_sql("PRAGMA table_info(customers);", conn)

,cid,name,type,notnull,dflt_value,pk
0,0,customerNumber,,0,None,0
1,1,customerName,,0,None,0
2,2,contactLastName,,0,None,0
3,3,contactFirstName,,0,None,0
4,4,phone,,0,None,0
5,5,addressLine1,,0,None,0
6,6,addressLine2,,0,None,0
7,7,city,,0,None,0
8,8,state,,0,None,0
9,9,postalCode,,0,None,0


In [5]:
pd.read_sql("PRAGMA table_info(orders);", conn)

,cid,name,type,notnull,dflt_value,pk
0,0,orderNumber,,0,None,0
1,1,orderDate,,0,None,0
2,2,requiredDate,,0,None,0
3,3,shippedDate,,0,None,0
4,4,status,,0,None,0
5,5,comments,,0,None,0
6,6,customerNumber,,0,None,0


In [6]:
pd.read_sql("PRAGMA table_info(orderDetails);", conn)

,cid,name,type,notnull,dflt_value,pk
0,0,orderNumber,,0,None,0
1,1,productCode,,0,None,0
2,2,quantityOrdered,,0,None,0
3,3,priceEach,,0,None,0
4,4,orderLineNumber,,0,None,0


In [7]:
pd.read_sql("""
SELECT
    MIN(orderDate) AS earliest_order,
    MAX(orderDate) AS latest_order,
    COUNT(*) AS total_orders
FROM orders;
""", conn)

,earliest_order,latest_order,total_orders
0,2003-01-06,2005-05-31,326


In [8]:

query = """
SELECT
    c.customerNumber,
    c.customerName,
    SUM(od.quantityOrdered * od.priceEach) AS total_spending,
    COUNT(DISTINCT o.orderNumber) AS order_count
FROM customers AS c
JOIN orders AS o
    ON c.customerNumber = o.customerNumber
JOIN orderdetails AS od
    ON o.orderNumber = od.orderNumber
WHERE o.orderDate BETWEEN
    date((SELECT MAX(orderDate) FROM orders), '-6 months')
    AND (SELECT MAX(orderDate) FROM orders)
GROUP BY c.customerNumber, c.customerName
HAVING COUNT(DISTINCT o.orderNumber) >= 3
ORDER BY total_spending DESC
LIMIT 5;
"""

pd.read_sql(query, conn)


,customerNumber,customerName,total_spending,order_count
0,141,Euro+ Shopping Channel,406226.92,12
1,124,Mini Gifts Distributors Ltd.,233157.99,8
2,323,"Down Under Souveniers, Inc",114460.72,3


In [9]:
pd.read_sql("""
SELECT COUNT(*) AS qualifying_customers
FROM (
    SELECT o.customerNumber
    FROM orders AS o
    WHERE o.orderDate BETWEEN
        date((SELECT MAX(orderDate) FROM orders), '-6 months')
        AND (SELECT MAX(orderDate) FROM orders)
    GROUP BY o.customerNumber
    HAVING COUNT(DISTINCT o.orderNumber) >= 3
);
""", conn)

,qualifying_customers
0,3
